In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import cross_val_score, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import RidgeCV
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.metrics import mean_absolute_error
import xgboost as xgb
import lightgbm as lgb
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import glob
import warnings
warnings.filterwarnings('ignore')

In [2]:
regressions = {
    'xgboost': Pipeline([
        ('scaler', StandardScaler()),
        ('xgb', xgb.XGBRegressor(
            n_estimators=1000, max_depth=6, learning_rate=0.1,
            subsample=0.8, colsample_bytree=0.8, random_state=42, verbosity=0
        ))
    ]),
    
    'lightGBM': Pipeline([
        ('scaler', StandardScaler()),
        ('lgb', lgb.LGBMRegressor(
            n_estimators=1000, max_depth=8, learning_rate=0.1,
            feature_fraction=0.8, bagging_fraction=0.8, random_state=42, verbosity=-1
        ))
    ]),

    'randomForest': Pipeline([
        ('scaler', StandardScaler()),
        ('rf', RandomForestRegressor(
            n_estimators=500, max_depth=15, min_samples_split=5,
            min_samples_leaf=2, random_state=42, n_jobs=-1
        ))
    ])
}


In [3]:
def test_regression(base_folder, regression_name, save=False, output_file=None, visual_only=False):

    files = sorted(glob.glob(base_folder))
    mae_table = {}
    model = regressions[regression_name]
    test_maes = []
    dfs = []

    for fold in range(1, 6):
        fold_folder = files[fold-1]
        print(f'\n Processing Fold {fold}: {fold_folder}')

        if visual_only:
            X_train = pd.read_csv(f"{fold_folder}/X_train_visual.csv").values
            X_test = pd.read_csv(f"{fold_folder}/X_test_visual.csv").values
        else:      
            X_train = pd.read_csv(f"{fold_folder}/X_train_combined.csv").values
            X_test = pd.read_csv(f"{fold_folder}/X_test_combined.csv").values
        y_train = pd.read_csv(f"{fold_folder}/train_predictions_with_metadata.csv")['target'].values
        y_test = pd.read_csv(f"{fold_folder}/test_predictions_with_metadata.csv")['target'].values
    
        test_metadata = pd.read_csv(f"{fold_folder}/test_predictions_with_metadata.csv")[['target', 'CENTROID_ID', 'LATNUM', 'LONGNUM']]
        
        print(f"  Loaded: {X_train.shape[0]} train, {X_test.shape[0]} test samples")
    
        model.fit(X_train, y_train)
        test_predictions = model.predict(X_test)
        test_mae = mean_absolute_error(y_test, test_predictions)
    
        mae_table[f'fold {fold}'] = test_mae
        test_maes.append(test_mae)
        print(f'{regression_name}: test_mae: {test_mae:.4f}')
    
        fold_df = test_metadata.copy()
        fold_df['prediction'] = test_predictions
        fold_df['error'] = np.abs(y_test - test_predictions)
        fold_df['fold'] = fold
        dfs.append(fold_df)
    
    std_err = np.std(test_maes) / np.sqrt(5)
    print(f'Average: {np.mean(list(mae_table.values()))} ± {std_err}')
        
    
    if save:
        df_combined = pd.concat(dfs, ignore_index=True)
        df_combined.to_csv(f'results/{output_file}', index=False)
        print(f'Data saved to {output_file}')

    return mae_table
        




## Test Regression Head (sh)

### Combined Features (Geo-encoded + Visual)

In [10]:
base_folder = 'results/split_spatialL_*_sh'
mae_table_sh_xgb = test_regression(base_folder, 'xgboost', save=True, output_file='df_sh_xgb.csv')
mae_table_sh_xgb

['results/split_spatialL_1_sh',
 'results/split_spatialL_2_sh',
 'results/split_spatialL_3_sh',
 'results/split_spatialL_4_sh',
 'results/split_spatialL_5_sh']


 Processing Fold 1: results/split_spatialL_1_loc_locenc
  Loaded: 10602 train, 2713 test samples
xgboost: test_mae: 0.1818

 Processing Fold 2: results/split_spatialL_2_loc_locenc
  Loaded: 10685 train, 2630 test samples
xgboost: test_mae: 0.1772

 Processing Fold 3: results/split_spatialL_3_loc_locenc
  Loaded: 10668 train, 2647 test samples
xgboost: test_mae: 0.1844

 Processing Fold 4: results/split_spatialL_4_loc_locenc
  Loaded: 10645 train, 2670 test samples
xgboost: test_mae: 0.1802

 Processing Fold 5: results/split_spatialL_5_loc_locenc
  Loaded: 10660 train, 2655 test samples
xgboost: test_mae: 0.1850
0.18172151730839073


In [12]:
mae_table_sh_lgb = test_regression(base_folder, 'lightGBM', save=True, output_file='df_sh_lgb.csv')
mae_table_sh_lgb


 Processing Fold 1: results/split_spatialL_1_sh
  Loaded: 10602 train, 2713 test samples
light_gbm: test_mae: 0.1819

 Processing Fold 2: results/split_spatialL_2_sh
  Loaded: 10685 train, 2630 test samples
light_gbm: test_mae: 0.1767

 Processing Fold 3: results/split_spatialL_3_sh
  Loaded: 10668 train, 2647 test samples
light_gbm: test_mae: 0.1820

 Processing Fold 4: results/split_spatialL_4_sh
  Loaded: 10645 train, 2670 test samples
light_gbm: test_mae: 0.1809

 Processing Fold 5: results/split_spatialL_5_sh
  Loaded: 10660 train, 2655 test samples
light_gbm: test_mae: 0.1837
Average: 0.1810708776254164


In [ ]:
mae_table_sh_rf = test_regression(base_folder, 'randomForest', save=True, output_file='df_sh_rf.csv')
mae_table_sh_rf

### Visual-only Features

In [12]:
base_folder = 'results/split_spatialL_*_sh'

mae_table_visual_lgb = test_regression(base_folder, 'lightGBM')
mae_table_visual_lgb

['results/split_spatialL_1_loc_locenc',
 'results/split_spatialL_2_loc_locenc',
 'results/split_spatialL_3_loc_locenc',
 'results/split_spatialL_4_loc_locenc',
 'results/split_spatialL_5_loc_locenc']


 Processing Fold 1: results/split_spatialL_1_loc_locenc
  Loaded: 10602 train, 2713 test samples
light_gbm: test_mae: 0.2134

 Processing Fold 2: results/split_spatialL_2_loc_locenc
  Loaded: 10685 train, 2630 test samples
light_gbm: test_mae: 0.2114

 Processing Fold 3: results/split_spatialL_3_loc_locenc
  Loaded: 10668 train, 2647 test samples
light_gbm: test_mae: 0.2155

 Processing Fold 4: results/split_spatialL_4_loc_locenc
  Loaded: 10645 train, 2670 test samples
light_gbm: test_mae: 0.2116

 Processing Fold 5: results/split_spatialL_5_loc_locenc
  Loaded: 10660 train, 2655 test samples
light_gbm: test_mae: 0.2191
Average: 0.21419448467637475


In [15]:
mae_table_visual_xgb = test_regression(base_folder, 'xgboost')
mae_table_visual_xgb


 Processing Fold 1: results/split_spatialL_1_loc_locenc
  Loaded: 10602 train, 2713 test samples
xgboost: test_mae: 0.2137

 Processing Fold 2: results/split_spatialL_2_loc_locenc
  Loaded: 10685 train, 2630 test samples
xgboost: test_mae: 0.2130

 Processing Fold 3: results/split_spatialL_3_loc_locenc
  Loaded: 10668 train, 2647 test samples
xgboost: test_mae: 0.2175

 Processing Fold 4: results/split_spatialL_4_loc_locenc
  Loaded: 10645 train, 2670 test samples
xgboost: test_mae: 0.2126

 Processing Fold 5: results/split_spatialL_5_loc_locenc
  Loaded: 10660 train, 2655 test samples
xgboost: test_mae: 0.2186
0.21508324470107493


## Test Regression Head (sh+siren)

In [4]:
base_folder = 'results/split_spatialL_*_sh_siren'
files = sorted(glob.glob(base_folder))
mae_table_sh_siren = {}
files

['results/split_spatialL_1_sh_siren',
 'results/split_spatialL_2_sh_siren',
 'results/split_spatialL_3_sh_siren',
 'results/split_spatialL_4_sh_siren',
 'results/split_spatialL_5_sh_siren']

In [ ]:
mae_table_sh_siren_lgb = test_regression(base_folder, 'lightGBM')
mae_table_visual_lgb


 Processing Fold 1: results/split_spatialL_1_sh_siren
  Loaded: 10602 train, 2713 test samples


## Cleaned Data (sh)

In [4]:
base_folder = 'results/split_spatialL_*_sh_cleaned'
mae_table_sh_cleaned_lgb = test_regression(base_folder, 'lightGBM', True, 'df_sh_cleaned_lgb.csv')
mae_table_sh_cleaned_lgb


 Processing Fold 1: results/split_spatialL_1_sh_cleaned
  Loaded: 10602 train, 2713 test samples
lightGBM: test_mae: 0.1841

 Processing Fold 2: results/split_spatialL_2_sh_cleaned
  Loaded: 10685 train, 2630 test samples
lightGBM: test_mae: 0.1772

 Processing Fold 3: results/split_spatialL_3_sh_cleaned
  Loaded: 10668 train, 2647 test samples
lightGBM: test_mae: 0.1822

 Processing Fold 4: results/split_spatialL_4_sh_cleaned
  Loaded: 10645 train, 2670 test samples
lightGBM: test_mae: 0.1757

 Processing Fold 5: results/split_spatialL_5_sh_cleaned
  Loaded: 10660 train, 2655 test samples
lightGBM: test_mae: 0.1852
Average: 0.180864293553001 ± 0.0016869726019737282
Data saved to df_sh_cleaned_lgb.csv


{'fold 1': 0.1840810447392199,
 'fold 2': 0.17717673953680976,
 'fold 3': 0.1821818983955736,
 'fold 4': 0.17569185307363694,
 'fold 5': 0.1851899320197648}

In [5]:
mae_table_sh_cleaned_xgb = test_regression(base_folder, 'xgboost', True, 'df_sh_cleaned_xgb.csv')
mae_table_sh_cleaned_xgb


 Processing Fold 1: results/split_spatialL_1_sh_cleaned
  Loaded: 10602 train, 2713 test samples
xgboost: test_mae: 0.1860

 Processing Fold 2: results/split_spatialL_2_sh_cleaned
  Loaded: 10685 train, 2630 test samples
xgboost: test_mae: 0.1771

 Processing Fold 3: results/split_spatialL_3_sh_cleaned
  Loaded: 10668 train, 2647 test samples
xgboost: test_mae: 0.1841

 Processing Fold 4: results/split_spatialL_4_sh_cleaned
  Loaded: 10645 train, 2670 test samples
xgboost: test_mae: 0.1777

 Processing Fold 5: results/split_spatialL_5_sh_cleaned
  Loaded: 10660 train, 2655 test samples
xgboost: test_mae: 0.1847
Average: 0.18193559325761383 ± 0.0016773095754479097
Data saved to df_sh_cleaned_xgb.csv


{'fold 1': 0.18602043547393418,
 'fold 2': 0.17712218857451267,
 'fold 3': 0.18411774447099985,
 'fold 4': 0.17769730769953745,
 'fold 5': 0.1847202900690849}

In [ ]:
mae_table_sh_cleaned_rf = test_regression(base_folder, 'randomForest', True, 'df_sh_cleaned_rf.csv')
mae_table_sh_cleaned_rf

### Visual-only

In [ ]:
mae_table_sh_visual_cleaned_lgb = test_regression(base_folder, 'randomForest', visual_only=True)
mae_table_sh_visual_cleaned_lgb